In [1]:
def generate_random_math_question():
    import random

    operations = ['+', '-', '*', '/']
    operation = random.choice(operations)

    if operation == '+':
        num1 = random.randint(1, 10)
        num2 = random.randint(1, 10)
        output = {
            "operation": operation,
            "a": num1,
            "b": num2,
            "answer": num1 + num2
        }
    elif operation == '-':
        num1 = random.randint(2, 10)
        num2 = random.randint(1, num1) # For multiplication, we keep it simple so num1 >= num2
        output = {
            "operation": operation,
            "a": num1,
            "b": num2,
            "answer": num1 - num2
        }
    elif operation == '*':
        num1 = random.randint(1, 10)
        num2 = random.randint(1, 10)
        output = {
            "operation": operation,
            "a": num1,
            "b": num2,
            "answer": num1 * num2
        }
    elif operation == '/':
        # Ensure no division by zero and integer result
        num2 = random.randint(1, 10)
        num1 = num2 * random.randint(1, 10)
        output = {
            "operation": operation,
            "a": num1,
            "b": num2,
            "answer": int(num1 / num2)
        }

    return output

In [2]:
generate_random_math_question()

{'operation': '/', 'a': 2, 'b': 2, 'answer': 1}

Vector Stores Integration

In [3]:
from pathlib import Path

from app.rag.vector_store import (
    get_chroma_client,
    get_collection,
    get_topic_collection,
    get_topic_registry,
    register_topic,
    topic_collection_name,
    topic_exists,
    reset_topic_registry
)
from app.rag.ingestion import rag_document_ingestion, vector_store_ingestion


In [4]:
client = get_chroma_client()
topic_registry = get_topic_registry()

print("Chroma client initialized.")
print("Topic registry name:", topic_registry.name)
print("Topic registry count:", topic_registry.count())


Chroma client initialized.
Topic registry name: topic_registry
Topic registry count: 0


In [5]:
topic = "football"
collection_name = topic_collection_name(topic)

football_collection = get_topic_collection(topic)

print("Topic collection initialized:", football_collection.name)
print("Topic exists already?", topic_exists(topic))


Topic collection initialized: topic_football
Topic exists already? None


In [6]:
textfile = "football_test.txt"

chunks, ids, metadata = rag_document_ingestion(
    textfile=textfile,
    topic=topic,
    chunk_size=300,
    chunk_overlap=50,
)

football_collection = vector_store_ingestion(
    topic=topic,
    chunks=chunks,
    ids=ids,
    metadata=metadata,
    vector_store_name=collection_name,
)

print("Inserted chunks:", len(chunks))
print("Collection count:", football_collection.count())
print("Topic registry lookup:", topic_exists(topic))


Topic 'football' not found in registry. Creating new collection.


Add of existing embedding ID: chunk_0
Add of existing embedding ID: chunk_1
Add of existing embedding ID: chunk_2
Add of existing embedding ID: chunk_3
Add of existing embedding ID: chunk_4
Add of existing embedding ID: chunk_5
Add of existing embedding ID: chunk_6
Add of existing embedding ID: chunk_7
Add of existing embedding ID: chunk_8
Add of existing embedding ID: chunk_9
Add of existing embedding ID: chunk_10
Add of existing embedding ID: chunk_11
Add of existing embedding ID: chunk_12
Add of existing embedding ID: chunk_13
Add of existing embedding ID: chunk_14
Add of existing embedding ID: chunk_15
Add of existing embedding ID: chunk_16
Add of existing embedding ID: chunk_17
Add of existing embedding ID: chunk_18
Add of existing embedding ID: chunk_19
Add of existing embedding ID: chunk_20
Add of existing embedding ID: chunk_21
Add of existing embedding ID: chunk_22
Add of existing embedding ID: chunk_23
Add of existing embedding ID: chunk_24
Add of existing embedding ID: chunk

Inserted chunks: 57
Collection count: 57


Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


Topic registry lookup: topic_football


In [7]:
stored = football_collection.get()

print("Stored ids:", stored["ids"][:5])
print("Stored metadata sample:", stored["metadatas"][:2])
print("First stored chunk:\n")
print(stored["documents"][0])


Stored ids: ['chunk_0', 'chunk_1', 'chunk_2', 'chunk_3', 'chunk_4']
Stored metadata sample: [{'topic': 'football'}, {'topic': 'football'}]
First stored chunk:

## Introduction to Football


In [8]:
question = "How long is a standard football match and how many players are on the field at the start?"

results = football_collection.query(
    query_texts=[question],
    n_results=3,
)

print("Top matching chunks:\n")
for i, doc in enumerate(results["documents"][0], start=1):
    print(f"--- Result {i} ---")
    print(doc)
    print()


Top matching chunks:

--- Result 1 ---
---

## Basic Structure of the Game

A football match is played between two teams, each consisting of **11 players**, making **22 players total** on the field at the start of the match. Each team usually consists of:

* 1 goalkeeper
* 4 defenders
* 3–5 midfielders
* 1–3 forwards

--- Result 2 ---
---

## Match Timing

A standard football match lasts **90 minutes**, divided into two halves:

* First half: 45 minutes
* Second half: 45 minutes

Between the halves there is a **15-minute halftime break**.

--- Result 3 ---
The match is played on a field called a **pitch**, which typically measures about **100–110 meters long** and **64–75 meters wide**. At each end of the pitch is a rectangular goal consisting of two vertical posts and a horizontal crossbar.



In [14]:
topic_exists("soccer", return_matches=True)


Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


[{'topic': 'football',
  'collection_name': 'topic_football',
  'similarity': 0.6739367862420614}]